# 12-3절 연습 문제 풀이

본문 [코드 12-13]을 바탕으로 연습 문제 12-9 ~ 12-11을 푼다.

> **NVIDIA GPU(CUDA) 환경이 필요하다.** `bitsandbytes` 양자화는 CUDA에
> 최적화되어 있다.

## 공통 준비

In [1]:
import sys
sys.path.append('../../')

from code_reference import common
from code_reference import visualize as viz

viz.configure(save_grayscale=False)
common.set_korean_plot_env()

SEED = 42
common.set_seed(SEED)
device = common.get_device()

import gc
import time
import torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
)

MODEL_NAME = 'Bllossom/llama-3.2-Korean-Bllossom-3B'
print(f'CUDA 사용 가능: {torch.cuda.is_available()}')

CUDA를 사용합니다.


/home/crapas/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA 사용 가능: True


In [2]:
# 참고 - 공통 헬퍼
def gpu_mb():
    return torch.cuda.memory_allocated() / 1024 ** 2


def unload(*names):
    """전역 이름을 비우고 가속기 캐시를 반환한다.

    함수 안에서 del 을 해도 호출한 쪽의 이름은 그대로 남아 객체가 살아 있다.
    그래서 이름 문자열을 받아 전역에서 직접 비운다.
    """
    g = globals()
    for n in names:
        if isinstance(n, str) and n in g:
            g[n] = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
terminators = [
    tokenizer.convert_tokens_to_ids('<|end_of_text|>'),
    tokenizer.convert_tokens_to_ids('<|eot_id|>'),
]


def chat_template(prompt):
    return [
        {'role': 'system',
         'content': '당신은 한국어를 사용하는 친절한 AI 친구입니다.'},
        {'role': 'user', 'content': prompt},
    ]


def ask(prompt, model, max_new_tokens=200, temperature=0.6, measure=False):
    input_ids = tokenizer.apply_chat_template(
        chat_template(prompt), add_generation_prompt=True,
        return_tensors='pt', return_dict=False,
    ).to(device)
    if measure:
        torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        output_ids = model.generate(
            input_ids, max_new_tokens=max_new_tokens,
            eos_token_id=terminators, do_sample=True,
            temperature=temperature, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    if measure:
        torch.cuda.synchronize()
    elapsed = time.time() - t0
    n_new = output_ids.shape[-1] - input_ids.shape[-1]
    text = tokenizer.decode(output_ids[0][input_ids.shape[-1]:],
                            skip_special_tokens=True).strip()
    return (text, elapsed, n_new) if measure else text

In [3]:
# 참고 - 양자화 설정으로 모델을 불러오고 메모리를 재는 함수
def load_quantized(bnb_config=None, label=''):
    gc.collect()
    torch.cuda.empty_cache()
    before = gpu_mb()
    t0 = time.time()
    kwargs = {'device_map': 'auto'}
    if bnb_config is not None:
        kwargs['quantization_config'] = bnb_config
    else:
        kwargs['dtype'] = torch.float16
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **kwargs)
    model.eval()
    mem = gpu_mb() - before
    print(f'[{label}] 로딩 {time.time() - t0:.1f}초, GPU 메모리 {mem:,.0f} MB')
    return model, mem

In [4]:
PROMPT = '대규모 언어 모델을 적은 자원으로 다루는 방법을 알려 줘.'
print(f'평가 프롬프트: {PROMPT}')

평가 프롬프트: 대규모 언어 모델을 적은 자원으로 다루는 방법을 알려 줘.


## 연습 문제 12-9

> [코드 12-13]에서 `bnb_4bit_quant_type='nf4'`를 `'fp4'`로 바꾸고 같은 프롬프트로
> 답변 품질을 비교해 보자. 두 형식 모두 4비트 양자화라 메모리 사용량은 거의
> 같으므로, 메모리도 함께 측정해 차이가 없음을 확인하고 짧은 답변에서는 미미할 수
> 있는 품질 차이에 주목한다.

In [5]:
results_9 = {}
for qtype in ('nf4', 'fp4'):
    cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type=qtype,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model, mem = load_quantized(cfg, qtype)
    common.set_seed(SEED)
    text, elapsed, n_new = ask(PROMPT, model, measure=True)
    results_9[qtype] = {'mem': mem, 'text': text,
                        'elapsed': elapsed, 'n_new': n_new}
    print(f'  생성 {n_new}토큰, {elapsed:.1f}초')
    unload('model')

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/254 [00:00<00:37,  6.83it/s]

Loading weights:   5%|▌         | 13/254 [00:00<00:03, 60.50it/s]

Loading weights:  13%|█▎        | 32/254 [00:00<00:02, 107.57it/s]

Loading weights:  20%|██        | 52/254 [00:00<00:01, 140.06it/s]

...

Loading weights:  86%|████████▌ | 219/254 [00:00<00:00, 264.48it/s]

Loading weights:  97%|█████████▋| 246/254 [00:01<00:00, 255.09it/s]

Loading weights: 100%|██████████| 254/254 [00:01<00:00, 232.59it/s]

[fp4] 로딩 2.1초, GPU 메모리 2,140 MB


  생성 200토큰, 14.6초


In [6]:
print(f'{"양자화 형식":<12} {"GPU 메모리(MB)":>16} {"생성 토큰":>10} {"생성 시간(초)":>14}')
print('-' * 58)
for q, r in results_9.items():
    print(f'{q:<12} {r["mem"]:>16,.0f} {r["n_new"]:>10} {r["elapsed"]:>14.1f}')
diff = abs(results_9['nf4']['mem'] - results_9['fp4']['mem'])
print(f'\n메모리 차이: {diff:,.1f} MB '
      f'({diff / results_9["nf4"]["mem"] * 100:.2f}%)')

양자화 형식            GPU 메모리(MB)      생성 토큰       생성 시간(초)
----------------------------------------------------------
nf4                     2,139        200           14.5
fp4                     2,140        200           14.6

메모리 차이: 1.4 MB (0.06%)


In [7]:
for q, r in results_9.items():
    print(f'===== {q} =====')
    print(r['text'][:400])
    print()

===== nf4 =====
대규모 언어 모델을 적은 자원으로 다루는 방법은 여러 가지로 나눌 수 있습니다. 아래는 몇 가지 방법을 소개합니다.

1. **GPU(그래픽 프로세서) 사용**: 대규모 언어 모델은 많은 컴퓨팅 자원을 필요로합니다. GPU를 사용하여 모델의 컴퓨팅 자원을 최적화할 수 있습니다. NVIDIA의 CUDA나 AMD의 ROCm 같은 GPU 소프트웨어를 사용하면 효과적으로 모델의 성능을 향상시킬 수 있습니다.

2. **Cloud Services**: AWS의 SageMaker, Google Cloud의 AutoML, Microsoft Azure의 Machine Learning Service 등은 대규모 언어 모델을 training하고 inference를 수행할 수 있는 cloud services입니다. 이 서비스를

===== fp4 =====
대규모 언어 모델을 적은 자원으로 다루는 방법에는 여러 가지가 있습니다. 아래는 몇 가지 방법입니다.

1. **가속도**: 대규모 언어 모델의 가속도를 줄이면 자원 사용량을 줄일 수 있습니다. 예를 들어, GPU 가속을 사용하여 가속도를 줄일 수 있습니다.
2. **모델 크기**: 대규모 언어 모델의 크기를 줄이면 자원 사용량을 줄일 수 있습니다. 예를 들어, smaller 모델을 사용하거나, 특정 task에만 모델을 사용할 수 있습니다.
3. **인ference time**: 대규모 언어 모델의 inference time(인퍼런스 시간)을 줄이면 자원 사용량을 줄일 수 있습니다. 예를 들어, 모델의 가속도를 줄이거나, 모델을 컴퓨터에 로드하는 시간을 줄이면 inference time을 줄일 수 있습니다.



In [8]:
# 짧은 답변에서는 차이가 잘 안 보이므로 여러 프롬프트로 넓혀 본다
PROBES = [
    '3 곱하기 17은 얼마야? 숫자만 답해 줘.',
    '대한민국의 수도는 어디야? 한 단어로 답해 줘.',
    '파이썬에서 리스트를 뒤집는 코드를 한 줄로 써 줘.',
    '"나는 학교에 간다"를 영어로 번역해 줘.',
]
probe_out = {}
for qtype in ('nf4', 'fp4'):
    cfg = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type=qtype,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model, _ = load_quantized(cfg, qtype)
    outs = []
    for p in PROBES:
        common.set_seed(SEED)
        outs.append(ask(p, model, max_new_tokens=60, temperature=0.3))
    probe_out[qtype] = outs
    unload('model')

for i, p in enumerate(PROBES):
    print(f'[질문] {p}')
    for qtype in ('nf4', 'fp4'):
        print(f'  {qtype}: {probe_out[qtype][i][:90]}')
    print()

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/254 [00:00<00:30,  8.18it/s]

Loading weights:  10%|▉         | 25/254 [00:00<00:01, 133.06it/s]

Loading weights:  20%|█▉        | 50/254 [00:00<00:01, 182.36it/s]

Loading weights:  30%|███       | 77/254 [00:00<00:00, 208.82it/s]

...

Loading weights:  78%|███████▊  | 197/254 [00:00<00:00, 239.37it/s]

Loading weights:  89%|████████▊ | 225/254 [00:01<00:00, 249.80it/s]

Loading weights: 100%|██████████| 254/254 [00:01<00:00, 220.77it/s]

[fp4] 로딩 2.1초, GPU 메모리 2,140 MB


[질문] 3 곱하기 17은 얼마야? 숫자만 답해 줘.
  nf4: 51
  fp4: 51

[질문] 대한민국의 수도는 어디야? 한 단어로 답해 줘.
  nf4: 서울
  fp4: 서울

[질문] 파이썬에서 리스트를 뒤집는 코드를 한 줄로 써 줘.
  nf4: 파이썬에서 리스트를 뒤집는 방법은 `reversed()` 함수를 사용하는 것입니다. 예를 들어:

```python
my_list = [1, 2, 3, 4, 5]
  fp4: 파이썬에서 리스트를 뒤집는 방법은 `reversed()` 함수를 사용하는 것입니다. 예를 들어, `my_list = [1, 2, 3, 4, 5]` 이면 `list

[질문] "나는 학교에 간다"를 영어로 번역해 줘.
  nf4: "나는 학교에 간다"를 영어로 번역해 드릴게요.

"나는 학교에 간다"는 영어로 "I'm going to school"로 번역됩니다.
  fp4: "나는 학교에 간다"는 영어로 "I am going to school"로 번역될 수 있습니다.



### 풀이 해설 — 연습 문제 12-9

**메모리는 사실상 같다.** 두 형식 모두 파라미터 하나를 4비트로 저장하므로
가중치가 차지하는 양이 같을 수밖에 없다. 위 표의 차이는 0.1% 안쪽이다.
지문이 "메모리도 함께 측정해 **차이가 없음을 확인**하고"라고 한 그대로다.

**차이는 격자점을 어디에 두느냐에 있다.** 본문 p22의 설명과 이어진다.

> 격자점을 가중치 구간 안에 **같은 간격으로** 배치하면 INT4 양자화 방식이 된다.
> 가중치 분포에 맞춰 **0 부근에 격자점을 촘촘히** 배치하면 NF4 양자화 방식이
> 되는데, LLM 가중치가 0 근처에 몰리는 경향을 반영한 것이다.

`fp4`는 4비트 부동소수점의 표현 가능한 값을 격자점으로 쓰고, `nf4`는 가중치가
정규 분포를 따른다는 가정 아래 **분위수로 격자점을 배치**한다. LLM 가중치가
실제로 0 근처에 몰려 있으므로 `nf4` 쪽이 같은 4비트로 더 촘촘하게 근사한다.

**그런데 짧은 답변에서는 차이가 잘 보이지 않는다.** 지문도 "짧은 답변에서는
**미미할 수 있는** 품질 차이에 주목한다"고 예고했다. 계산이나 번역처럼 **정답이
하나인 짧은 질문**을 여러 개 던지면 그나마 차이가 드러날 여지가 생긴다.

**본문의 권고와 같은 결론에 이른다.**

> 길고 정밀한 답변이 필요한 코딩이나 수학 추론 등의 작업에서는 큰 차이가 발생할
> 수 있으므로 양자화를 적용하지 않거나 8비트 양자화를 쓰는 것이 안전하다.

**적절성: 좋다.** 지문이 "메모리는 같다"는 결론을 미리 알려 주고 **품질에
주목하라고 방향을 잡아 준 것**이 특히 좋다. 이 단서가 없으면 독자가 메모리
숫자만 보고 "차이가 없네"로 끝낼 수 있다.

## 연습 문제 12-10

> [코드 12-13]에서 `load_in_4bit=True` 대신 `load_in_8bit=True`로 양자화 모델의
> 메모리와 답변 품질을 4비트 결과와 비교해 보자. 또한 8비트 모드에서
> `BitsAndBytesConfig`의 4비트 관련 인자가 어떻게 처리되는지 확인하고, 8비트에서
> 의미가 없는 인자를 정리해 더욱 깔끔한 코드로 만들어 보자.

In [9]:
# 8비트 설정에 4비트 인자를 그대로 남겨 두면 어떻게 되는지 먼저 확인한다
cfg_8bit_dirty = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_4bit_quant_type='nf4',              # 4비트 전용 인자
    bnb_4bit_compute_dtype=torch.float16,   # 4비트 전용 인자
    bnb_4bit_use_double_quant=True,         # 4비트 전용 인자
)
print('8비트 설정 객체에 4비트 인자를 함께 넘겼을 때 보관되는 값')
d = cfg_8bit_dirty.to_dict()
for k in sorted(d):
    if k.startswith('bnb_') or k.startswith('load_in'):
        print(f'  {k:<32} {d[k]}')

8비트 설정 객체에 4비트 인자를 함께 넘겼을 때 보관되는 값
  bnb_4bit_compute_dtype           float16
  bnb_4bit_quant_storage           uint8
  bnb_4bit_quant_type              nf4
  bnb_4bit_use_double_quant        True
  load_in_4bit                     False
  load_in_8bit                     True


In [10]:
# 8비트에서 의미가 있는 인자만 남긴 깔끔한 설정
cfg_8bit = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,          # 이상치로 볼 활성값의 절댓값 기준
    llm_int8_skip_modules=None,      # 양자화에서 제외할 모듈
)
print('8비트에서 의미가 있는 인자')
d8 = cfg_8bit.to_dict()
for k in sorted(d8):
    if k.startswith('llm_int8') or k.startswith('load_in'):
        print(f'  {k:<32} {d8[k]}')

8비트에서 의미가 있는 인자
  llm_int8_enable_fp32_cpu_offload False
  llm_int8_has_fp16_weight         False
  llm_int8_skip_modules            None
  llm_int8_threshold               6.0
  load_in_4bit                     False
  load_in_8bit                     True


In [11]:
# FP16(양자화 없음) / 8비트 / 4비트(NF4) 세 가지를 나란히 비교한다
configs = [
    ('FP16(양자화 없음)', None),
    ('INT8', cfg_8bit),
    ('NF4', BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True)),
]
results_10 = {}
for label, cfg in configs:
    model, mem = load_quantized(cfg, label)
    common.set_seed(SEED)
    text, elapsed, n_new = ask(PROMPT, model, measure=True)
    results_10[label] = {'mem': mem, 'text': text,
                         'elapsed': elapsed, 'n_new': n_new}
    print(f'  생성 {n_new}토큰, {elapsed:.1f}초 '
          f'({n_new / elapsed:.1f}토큰/초)')
    unload('model')

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/254 [00:00<01:27,  2.89it/s]

Loading weights:  13%|█▎        | 32/254 [00:00<00:02, 88.74it/s]

Loading weights:  22%|██▏       | 57/254 [00:00<00:01, 129.27it/s]

Loading weights:  33%|███▎      | 84/254 [00:00<00:01, 159.87it/s]

...

Loading weights:  83%|████████▎ | 210/254 [00:00<00:00, 245.07it/s]

Loading weights:  93%|█████████▎| 235/254 [00:01<00:00, 236.28it/s]

Loading weights: 100%|██████████| 254/254 [00:01<00:00, 218.32it/s]

[NF4] 로딩 2.2초, GPU 메모리 2,140 MB


  생성 200토큰, 13.9초 (14.4토큰/초)


In [12]:
base = results_10['FP16(양자화 없음)']['mem']
print(f'{"설정":<18} {"GPU 메모리(MB)":>16} {"FP16 대비":>10} '
      f'{"토큰/초":>10}')
print('-' * 60)
for label, r in results_10.items():
    print(f'{label:<18} {r["mem"]:>16,.0f} {r["mem"] / base:>9.2f}x '
          f'{r["n_new"] / r["elapsed"]:>10.1f}')

설정                      GPU 메모리(MB)    FP16 대비       토큰/초
------------------------------------------------------------
FP16(양자화 없음)                  6,128      1.00x       28.5
INT8                          3,444      0.56x        7.3
NF4                           2,140      0.35x       14.4


In [13]:
for label, r in results_10.items():
    print(f'===== {label} =====')
    print(r['text'][:350])
    print()

===== FP16(양자화 없음) =====
대규모 언어 모델을 적은 자원으로 다루는 것은 매우 어려운 일입니다. 대규모 언어 모델은 수천 GB의 데이터와 수백 GB의 메모리를 필요로 하며, 이를 적은 자원으로 처리하는 것은 매우 복잡하고 시간이 걸릴 수 있습니다. 하지만 몇 가지 방법을 제안할 수 있습니다:

1. **분산 처리**: 대규모 언어 모델을 분산 처리하여 여러 컴퓨터나 클라우드 인스턴스를 사용하여 처리할 수 있습니다. 이를 통해 각 컴퓨터는 더 적은 자원을 사용하여 모델을 처리할 수 있습니다.

2. **모델 조정**: 모델의 크기를 줄이고, 더 적은 자원을 사용하도록 조정할 수 있습니다. 이는 모델의 성능을 줄일 수 있지만, 적은 자원을 사용할 수

===== INT8 =====
대규모 언어 모델을 적은 자원으로 다루는 것은 매우 어려운 일입니다. 대규모 언어 모델은 수십 GB 이상의 데이터와 고성능 컴퓨터가 필요합니다. 그러나 몇 가지 방법으로 대규모 언어 모델을 적은 자원으로 사용할 수 있습니다:

1. **클라우드 서비스 사용**: Amazon Web Services (AWS), Google Cloud Platform (GCP), Microsoft Azure 같은 클라우드 서비스를 사용하여 대규모 언어 모델을 호스팅하고 처리할 수 있습니다. 클라우드 서비스는 자원 관리와 비용 관리를 도와줍니다.

2. **가상 머신(Virtual Machine) 사용**: 가상 머신을 사용하여 대규모 언어

===== NF4 =====
대규모 언어 모델을 적은 자원으로 다루는 방법은 여러 가지로 나눌 수 있습니다. 아래는 몇 가지 방법을 소개합니다.

1. **GPU(그래픽 프로세서) 사용**: 대규모 언어 모델은 많은 컴퓨팅 자원을 필요로합니다. GPU를 사용하여 모델의 컴퓨팅 자원을 최적화할 수 있습니다. NVIDIA의 CUDA나 AMD의 ROCm 같은 GPU 소프트웨어를 사용하면 효과적으로 모델의 성능을 향상시킬 수 있습니다.

2. **Cloud 

### 풀이 해설 — 연습 문제 12-10

**먼저 지문의 두 번째 요구부터.** 8비트 모드에서 4비트 인자는 **오류 없이 그냥
보관만 된다.** `BitsAndBytesConfig`가 값을 검증하지 않기 때문이다. 동작에는
영향이 없지만 **코드를 읽는 사람을 헷갈리게 한다.** 8비트에서 의미가 있는 인자는
`llm_int8_` 로 시작하는 것들이다.

| 인자 | 4비트 | 8비트 |
|---|---|---|
| `bnb_4bit_quant_type` | ○ | **무시** |
| `bnb_4bit_compute_dtype` | ○ | **무시** |
| `bnb_4bit_use_double_quant` | ○ | **무시** |
| `llm_int8_threshold` | 무시 | ○ |
| `llm_int8_skip_modules` | 무시 | ○ |

`llm_int8_threshold`는 8비트 양자화의 핵심 장치다. **활성값 중 절댓값이 큰
이상치는 INT8로 누르지 않고 FP16으로 따로 계산한다.** 이상치 몇 개가 전체 정밀도를
망치는 것을 막는 방식이라, 4비트에는 없는 개념이다.

**메모리는 [표 12-8]의 예상과 맞는다.** FP16을 1로 놓으면 INT8이 약 절반,
NF4가 약 4분의 1 근처다. 정확히 0.5와 0.25가 아닌 것은 **양자화하지 않는 계층과
부수 정보** 때문이며, 본문 p23이 "이 계층의 비중은 모델에 따라 30~40%에 이르기도
한다"고 밝힌 그대로다.

**속도는 메모리와 반대로 움직인다.** 토큰/초를 보면 양자화할수록 느려지는
경향이 나타난다. 본문 p23의 설명과 같다.

> bitsandbytes의 4비트 양자화는 계산 단계에서 가중치를 FP16으로 다시 펼쳐
> 곱셈을 수행하기 때문에, 메모리는 크게 줄지만 속도 이득은 거의 없다. 오히려
> 양자화하지 않은 FP16 추론보다 느릴 수 있다.

**양자화는 속도가 아니라 메모리를 위한 기법**이라는 점이 수치로 확인된다.

**적절성: 매우 좋다.** 메모리 비교라는 뻔한 요구에 더해 **"의미 없는 인자를
정리해 깔끔한 코드로 만들어 보자"**를 붙인 것이 좋다. 라이브러리가 조용히
무시하는 인자를 찾아내는 일은 실무에서 자주 필요한데, 그 습관을 들이게 한다.

## 연습 문제 12-11 [도전 문제]

> [코드 12-13]의 `bnb_4bit_compute_dtype`을 `torch.bfloat16`과 `torch.float32`로
> 각각 바꿔 모델을 불러온 후, `torch.float16`일 때와 추론 속도·메모리·답변 품질을
> 비교해 보자(cuda 환경 필요). 가중치 저장 형식(4비트)과 실제 행렬 곱 연산 정밀도가
> 왜 다를 수 있는지, 그리고 연산 정밀도를 높이거나 낮추면 어떤 대가가 따르는지
> 정리해 보자.

In [14]:
DTYPES = [
    ('float16', torch.float16),
    ('bfloat16', torch.bfloat16),
    ('float32', torch.float32),
]
results_11 = {}
for label, dt in DTYPES:
    cfg = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=dt,
        bnb_4bit_use_double_quant=True,
    )
    model, mem = load_quantized(cfg, f'compute_dtype={label}')
    # 워밍업 후 측정
    common.set_seed(SEED)
    _ = ask('안녕?', model, max_new_tokens=8)
    common.set_seed(SEED)
    text, elapsed, n_new = ask(PROMPT, model, measure=True)
    results_11[label] = {'mem': mem, 'text': text,
                         'elapsed': elapsed, 'n_new': n_new}
    print(f'  생성 {n_new}토큰, {elapsed:.1f}초 '
          f'({n_new / elapsed:.1f}토큰/초)')
    unload('model')

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/254 [00:00<00:32,  7.85it/s]

Loading weights:   9%|▊         | 22/254 [00:00<00:02, 112.37it/s]

Loading weights:  18%|█▊        | 45/254 [00:00<00:01, 160.76it/s]

Loading weights:  28%|██▊       | 71/254 [00:00<00:00, 195.99it/s]

...

Loading weights:  76%|███████▌  | 193/254 [00:00<00:00, 260.30it/s]

Loading weights:  88%|████████▊ | 224/254 [00:00<00:00, 273.65it/s]

Loading weights: 100%|██████████| 254/254 [00:01<00:00, 244.87it/s]

[compute_dtype=float32] 로딩 1.9초, GPU 메모리 2,140 MB


  생성 200토큰, 14.0초 (14.3토큰/초)


In [15]:
print(f'{"compute_dtype":<16} {"GPU 메모리(MB)":>16} {"생성 토큰":>10} '
      f'{"토큰/초":>10} {"FP16 대비 속도":>15}')
print('-' * 74)
base_tps = results_11['float16']['n_new'] / results_11['float16']['elapsed']
for label, r in results_11.items():
    tps = r['n_new'] / r['elapsed']
    print(f'{label:<16} {r["mem"]:>16,.0f} {r["n_new"]:>10} '
          f'{tps:>10.1f} {tps / base_tps:>14.2f}x')

compute_dtype         GPU 메모리(MB)      생성 토큰       토큰/초      FP16 대비 속도
--------------------------------------------------------------------------
float16                     2,140        200       13.6           1.00x
bfloat16                    2,140        200       16.9           1.24x
float32                     2,140        200       14.3           1.05x


In [16]:
for label, r in results_11.items():
    print(f'===== compute_dtype = {label} =====')
    print(r['text'][:300])
    print()

===== compute_dtype = float16 =====
대규모 언어 모델을 적은 자원으로 다루는 방법은 여러 가지로 나눌 수 있습니다. 아래는 몇 가지 방법을 소개합니다.

1. **GPU(그래픽 프로세서) 사용**: 대규모 언어 모델은 많은 컴퓨팅 자원을 필요로합니다. GPU를 사용하여 모델의 컴퓨팅 자원을 최적화할 수 있습니다. NVIDIA의 CUDA나 AMD의 ROCm 같은 GPU 소프트웨어를 사용하면 효과적으로 모델의 성능을 향상시킬 수 있습니다.

2. **Cloud Services**: AWS의 SageMaker, Google Cloud의 AutoML, Microsoft Az

===== compute_dtype = bfloat16 =====
대규모 언어 모델을 적은 자원으로 다루는 방법은 여러 가지로 나눌 수 있습니다. 아래는 몇 가지 방법을 소개합니다.

1. **GPU(그래픽 프로세서) 사용**: 대규모 언어 모델은 많은 컴퓨팅 자원을 필요로합니다. GPU를 사용하여 모델의 컴퓨팅 자원을 효율적으로 사용할 수 있습니다. NVIDIA의 CUDA 또는 AMD의 ROCm와 같은 GPU 소프트웨어를 사용하여 모델의 성능을 향상시킬 수 있습니다.

2. **Cloud Services**: cloud services를 사용하여 대규모 언어 모델을 학습하고 배포할 수 있습니다.

===== compute_dtype = float32 =====
대규모 언어 모델을 적은 자원으로 다루는 방법에는 여러 가지가 있습니다. 아래는 몇 가지 방법입니다:

1. **가속도**: 대규모 언어 모델은 매우 가속적인 성능을 제공합니다. 따라서, CPU와 GPU의 가속성을 높이면 모델의 성능을 크게 향상시킬 수 있습니다. 예를 들어, NVIDIA의 GeForce RTX 3080 GPU나 AMD의 Radeon RX 6800 XT GPU를 사용하면 성능을 크게 향상시킬 수 있습니다.

2. **메모리**: 대규모 언어 모델은 매우 많은 메모리를 필요로합니다. 따라서, 고

In [17]:
# 세 자료형이 표현할 수 있는 범위와 정밀도를 직접 확인한다
print(f'{"자료형":<12} {"비트":>5} {"지수부":>7} {"가수부":>7} '
      f'{"최대값":>14} {"최소 양수":>14}')
print('-' * 68)
for label, dt in DTYPES:
    fi = torch.finfo(dt)
    exp_bits = {torch.float16: 5, torch.bfloat16: 8, torch.float32: 8}[dt]
    man_bits = {torch.float16: 10, torch.bfloat16: 7, torch.float32: 23}[dt]
    print(f'{label:<12} {fi.bits:>5} {exp_bits:>7} {man_bits:>7} '
          f'{fi.max:>14.3e} {fi.tiny:>14.3e}')

자료형             비트     지수부     가수부            최대값          최소 양수
--------------------------------------------------------------------
float16         16       5      10      6.550e+04      6.104e-05
bfloat16        16       8       7      3.390e+38      1.175e-38
float32         32       8      23      3.403e+38      1.175e-38


### 풀이 해설 — 연습 문제 12-11

**지문의 첫 질문부터. 저장 형식과 연산 정밀도가 왜 다른가.**

4비트로 누른 가중치는 **정수 번호**일 뿐이라 그대로 곱셈할 수 없다. 그래서
`bitsandbytes`는 행렬 곱 직전에 **가중치를 `bnb_4bit_compute_dtype`으로 펼친 뒤**
곱하고, 펼친 값은 곧바로 버린다. 저장은 4비트로 유지하면서 계산만 부동소수점으로
하는 것이다. 본문의 설명 그대로다.

> `bnb_4bit_compute_dtype=torch.float16`은 가중치는 4비트로 두되 **행렬 곱
> 단계에서만 FP16으로 펼쳐** 정확도 손실을 줄이는 옵션이다.

**그래서 메모리는 거의 변하지 않는다.** 펼친 값이 오래 머물지 않기 때문이다.
위 표에서 세 설정의 메모리가 비슷한 것이 그 증거다.

**세 자료형의 성격은 마지막 표가 보여 준다.**

| | 비트 | 지수부 | 가수부 | 성격 |
|---|---|---|---|---|
| `float16` | 16 | 5 | 10 | **정밀도**가 높고 표현 범위가 좁다 |
| `bfloat16` | 16 | **8** | 7 | 범위가 `float32`와 같고 **정밀도가 낮다** |
| `float32` | 32 | 8 | 23 | 둘 다 넉넉하지만 **느리다** |

12장 도입부 박스가 BF16을 이렇게 소개한 것과 정확히 맞물린다.

> 같은 16비트지만 표현할 수 있는 값의 범위가 FP16보다 훨씬 넓고(대신 소수점
> 이하 정밀도는 조금 낮다), 큰 모델 학습 중 값이 폭주하거나 사라지는 문제가 적어
> 자주 쓰인다.

**★ 그런데 속도 측정 결과가 예상을 빗나간다.**

"정밀도를 올리면 느려진다"가 상식적인 예측이지만, 위 표는 그렇게 나오지 않았다.
**`bfloat16`이 가장 빠르고, `float32`가 `float16`보다도 조금 빨랐다.**

이유를 나눠 보면 이렇다.

- **`float32`가 느려지지 않는 이유** — `compute_dtype`은 **가중치를 펼칠 때
  쓰는 자료형**일 뿐, 모델 전체를 FP32로 돌리라는 뜻이 아니다. 입력 활성값은
  여전히 반정밀도이고, 병목도 **곱셈 자체가 아니라 4비트 가중치를 펼치는
  역양자화 단계**에 있다. 그래서 곱셈 정밀도를 올려도 전체 시간이 크게 늘지
  않는다.
- **`bfloat16`이 빠른 이유** — 최신 GPU의 텐서 코어에서 BF16과 FP16의 처리량은
  같다. 여기에 `bitsandbytes`의 역양자화 경로가 BF16에 더 잘 맞물린 것으로
  보인다. 측정값 하나로 단정하기는 어렵고, 생성은 실행마다 길이와 시간이 조금씩
  달라지므로 **여러 번 재서 평균을 봐야 한다.**

**메모리는 셋 다 2,140MB로 완전히 같다.** 이것이 오히려 확실한 결론이다.
펼친 값이 곧바로 버려지므로 `compute_dtype`은 메모리에 영향을 주지 않는다.

**그렇다면 무엇을 기준으로 고를 것인가.**

- **`float16`** — 표현 범위가 좁지만(최대 약 6.5×10⁴) 가수부가 10비트로 가장
  촘촘하다. 추론은 학습과 달리 값이 폭주할 일이 적어 기본값으로 무난하다.
- **`bfloat16`** — 범위가 `float32`와 같아 **오버플로 위험이 낮다.** 큰 모델이나
  긴 입력에서 안전하고, 이 측정에서는 속도도 가장 좋았다. 대신 가수부가 7비트라
  미세한 차이를 덜 구분한다.
- **`float32`** — 가장 안정적이지만 **얻는 것이 거의 없다.** 4비트로 누른 값의
  정밀도는 이미 16단계 격자에 묶여 있어, **연산 정밀도를 올려도 잃어버린 정보가
  돌아오지 않기 때문**이다.

**마지막 문장이 이 문제의 핵심이다.** 양자화로 잃은 정보는 연산 정밀도로
되살릴 수 없다. `compute_dtype`은 **추가 손실을 막는 장치**일 뿐이다.

**마지막 문장이 이 문제의 핵심이다.** 양자화로 잃은 정보는 연산 정밀도로
되살릴 수 없다. `compute_dtype`은 **추가 손실을 막는 장치**일 뿐이다.

**적절성: 매우 좋다.** 도전 문제답게 "측정해 보라"에서 그치지 않고 **"왜
다를 수 있는지 정리해 보자"**로 개념을 묻는다. 12장 도입부의 자료형 표기법 박스
(FP16, BF16, NF4)가 여기서 실제 선택지로 돌아온다는 점도 좋다.

---